# The Resilient Distributed Dataset (RDD)




## 1. SparkContext

A `SparkConf` holds the app settings. A `SparkContext` (`sc`) is the handle to Spark.

- **appName:** shown in the Spark UI and logs
- **master:** where to run. `local` = this process (one thread). The Docker cluster is a different master (`spark://spark-master:7077`); we do not use it here.


In [1]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setAppName("rdd-lesson").setMaster("local")
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/16 20:45:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Create an RDD

`parallelize` copies a Python list into an RDD.


In [2]:
data = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
dataDist = sc.parallelize(data)
type(dataDist)

pyspark.rdd.RDD

## 3. Actions

Transformations are lazy. **Actions** run the job and return a Python value.

`collect()` pulls **every** element to the driver. Fine for 10 numbers. Never do this on the full CDC file.


In [3]:
dataList = dataDist.collect()
print(type(dataList))
print(dataList)

<class 'list'>
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


`take(n)` returns the first *n* elements. Safe on large data.


In [4]:
print(dataDist.take(3))
print(dataDist.takeOrdered(2))
print(dataDist.takeSample(False, 3))  # False = without replacement

[0, 1, 2]
[0, 1]
[5, 7, 6]


`count`, `countByValue`, `top`, `max`, `min`.


In [5]:
print(dataDist.count())
print(dict(dataDist.countByValue()))
print(dataDist.top(5))
print(dataDist.max(), dataDist.min())

10
{0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
[9, 8, 7, 6, 5]
9 0


## 4. Transformations: `map` and `reduce`

**Map** applies a function to each element. **Reduce** folds two values at a time into one. Both can take a `def` or a `lambda`.


In [6]:
def compute_pow(d):
    return d * d

powDist = dataDist.map(compute_pow)
powDist.collect()

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

In [7]:
powDist = dataDist.map(lambda d: d * d)
powDist.collect()

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

In [8]:
def add(a, b):
    return a + b

print(dataDist.reduce(add))
print(dataDist.reduce(lambda a, b: a + b))

45
45


### `filter`

Keeps elements for which the function is true.


In [11]:
words = [
    "Artificial Intelligence",
    "Machine Learning",
    "Reinforcement Learning",
    "Deep Learning",
    "Computer Vision",
    "Natural Language Processing",
    "Augmented Reality",
    "Blockchain",
    "Robotic",
    "Cyber Security",
]
wordsDist = sc.parallelize(words)
print(wordsDist.filter(lambda w: len(w) > 15))
print(wordsDist.filter(lambda w: len(w) > 15).collect())
print(wordsDist.filter(lambda w: w[0].lower() in "aeiou").collect())

PythonRDD[18] at RDD at PythonRDD.scala:53
['Artificial Intelligence', 'Machine Learning', 'Reinforcement Learning', 'Natural Language Processing', 'Augmented Reality']
['Artificial Intelligence', 'Augmented Reality']


## 5. Combining RDDs

`union` concatenates. `subtract` keeps values in the first RDD that are not in the second.


In [12]:
dist1 = sc.parallelize([1, 2, 3, 4, 5])
dist2 = sc.parallelize([5, 6, 7, 8, 9])

print("union:", dist1.union(dist2).collect())
print("subtract:", dist1.subtract(dist2).collect())

union: [1, 2, 3, 4, 5, 5, 6, 7, 8, 9]
subtract: [2, 4, 1, 3]


### Wide transformations

These need data from more than one partition (a shuffle). Partitions themselves: next session.

`intersection` = in both. `distinct` = unique values. `cartesian` = all pairs.


In [13]:
print("intersection:", dist1.intersection(dist2).collect())
print("cartesian:", dist1.cartesian(dist2).collect())

namesDist = sc.parallelize(["Giuseppe", "Francesco", "Antonio", "Antonio", "Giuseppe"])
print("distinct:", namesDist.distinct().collect())

intersection: [5]
cartesian: [(1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (4, 5), (4, 6), (4, 7), (4, 8), (4, 9), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9)]
distinct: ['Giuseppe', 'Francesco', 'Antonio']


### Key–value: `sortByKey` and `join`

`join` returns `(k, (v1, v2))` for matching keys.


In [14]:
pairRDD = sc.parallelize([(1, 5), (1, 10), (2, 4), (3, 1), (2, 6)])
print(pairRDD.sortByKey().collect())

rdd1 = sc.parallelize([("a", 1), ("b", 4)])
rdd2 = sc.parallelize([("a", 2), ("a", 3)])
print(sorted(rdd1.join(rdd2).collect()))

[(1, 5), (1, 10), (2, 4), (2, 6), (3, 1)]
[('a', (1, 2)), ('a', (1, 3))]


# 6. Real data: CDC mortality 2021

CDC / NCHS *Multiple Cause-of-Death* public-use file: one **fixed-width** line per US death certificate. No header, no commas. Fields sit at column positions ([record layout](https://www.cdc.gov/nchs/data/dvs/Multiple-Cause-Record_Layout_2021.pdf)).

**In class we use a ~40,000-line sample** (every 87th record, months still look like 2021). The full file is 3.47 million lines / 2.6 GB (`mort2021us.txt`). Same slices. Do not `collect()` the full file.

| field | columns (1-based) | Python slice | codes |
| --- | --- | --- | --- |
| month of death | 65–66 | `[64:66]` | `01`–`12` |
| sex | 69 | `[68]` | `M` / `F` |
| age (years) | 71–73 | `[70:73]` | 999 = not stated |
| underlying cause | 146–149 | `[145:149]` | ICD-10, e.g. `U071` COVID-19 |


In [15]:
from pathlib import Path

here = Path.cwd()
candidates = [
    Path("/opt/spark/data/cdc/mort2021us_sample.txt"),
    here / "cdc" / "mort2021us_sample.txt",
    here.parent / "book_data" / "cdc" / "mort2021us_sample.txt",
    here / "book_data" / "cdc" / "mort2021us_sample.txt",
]
CDC_PATH = str(next(p for p in candidates if p.exists()))
print(CDC_PATH)

lines = sc.textFile(CDC_PATH)
print(type(lines))
lines.take(2)

/opt/spark/data/cdc/mort2021us_sample.txt
<class 'pyspark.rdd.RDD'>


['                  11                                          7101  F1080 422210  4D1                2021U7CN                                    C851129 039   13 0511I509 21I518 31I513 41C851 61M481                                                                                                                                              05 C851 I509 I513 I518 M481                                                                                                                    100  01                                                                                                                                                                                                                                                                                                                           184005949020',
 '                  22                                          3101  M1070 402009  1S3                2021U7BN                                    J690273 088   37 0611I469 21E87

Count the lines. Instant on the sample; a real job on the 2.6 GB file.


In [16]:
lines.count()

39910

### Parse with `map`

`map` is lazy. `take` / `count` run the job.


In [17]:
def parse(row):
    return {
        "month": row[64:66],
        "sex": row[68],
        "age": int(row[70:73]),
        "cause": row[145:149].strip(),
    }

deaths = lines.map(parse)
deaths.take(5)

[{'month': '01', 'sex': 'F', 'age': 80, 'cause': 'C851'},
 {'month': '01', 'sex': 'M', 'age': 70, 'cause': 'J690'},
 {'month': '01', 'sex': 'M', 'age': 92, 'cause': 'F03'},
 {'month': '01', 'sex': 'M', 'age': 46, 'cause': 'J80'},
 {'month': '01', 'sex': 'M', 'age': 38, 'cause': 'X45'}]

### `filter`
Women, then COVID-19 (`U071`).


In [18]:
women = deaths.filter(lambda d: d["sex"] == "F")
print("women (sample):", women.count())
women.take(3)

women (sample): 18610


[{'month': '01', 'sex': 'F', 'age': 80, 'cause': 'C851'},
 {'month': '01', 'sex': 'F', 'age': 83, 'cause': 'C349'},
 {'month': '03', 'sex': 'F', 'age': 72, 'cause': 'I250'}]

In [19]:
covid = deaths.filter(lambda d: d["cause"] == "U071")
print("COVID-19 (sample):", covid.count())
covid.take(3)

COVID-19 (sample): 4724


[{'month': '08', 'sex': 'F', 'age': 44, 'cause': 'U071'},
 {'month': '08', 'sex': 'F', 'age': 90, 'cause': 'U071'},
 {'month': '09', 'sex': 'F', 'age': 81, 'cause': 'U071'}]

### `countByValue`
Same action as on `[0, 1, …, 9]`.


In [20]:
dict(deaths.map(lambda d: d["sex"]).countByValue())

{'F': 18610, 'M': 21300}

In [21]:
by_month = deaths.map(lambda d: d["month"]).countByValue()
sorted(by_month.items())

[('01', 4265),
 ('02', 3290),
 ('03', 3081),
 ('04', 2942),
 ('05', 2938),
 ('06', 2858),
 ('07', 3014),
 ('08', 3448),
 ('09', 3592),
 ('10', 3479),
 ('11', 3340),
 ('12', 3663)]

### `map` + `reduce`
COVID count: map each record to `1` or `0`, then add.


In [22]:
covid_n = deaths.map(lambda d: 1 if d["cause"] == "U071" else 0).reduce(lambda a, b: a + b)
print(covid_n)

4724


Top causes: map to the ICD code, `countByValue`, sort.


In [23]:
causes = deaths.map(lambda d: d["cause"]).countByValue()
sorted(causes.items(), key=lambda kv: -kv[1])[:10]

[('U071', 4724),
 ('I251', 1861),
 ('C349', 1495),
 ('G309', 1235),
 ('J449', 1214),
 ('I219', 1181),
 ('F03', 1082),
 ('I250', 886),
 ('I500', 710),
 ('I64', 606)]

### Narrow vs wide: time the difference

On 40k rows both finish in a blink. Copy each record **15 times** (~600k) and **cache**, so we measure CPU + shuffle, not the text-file read.

- **Narrow** (`filter`, `map`): each record stays in its partition. One stage.
- **Wide** (`join` on a unique id): Spark must **shuffle** both sides so matching keys land together. Same *N* output rows — the extra time is the shuffle, not a combinatorial explosion.

`groupByKey` on cause is a bad demo here: only ~1,300 keys, so the gap is tiny. A 1-to-1 `join` makes the shuffle obvious. Watch [Spark UI](http://localhost:4040): narrow is one stage; the join has shuffle write / shuffle read.

In [ ]:
import time

repeated = deaths.flatMap(lambda d: [d] * 15).cache()
print("cached rows:", repeated.count())


def timed(label, fn):
    t0 = time.perf_counter()
    result = fn()
    print(f"{label}: {time.perf_counter() - t0:.2f}s  -> {result}")


def run_narrow():
    return (
        repeated.filter(lambda d: d["age"] != 999)
        .map(lambda d: d["age"] * d["age"] + hash(d["cause"]) % 97)
        .filter(lambda score: score > 0)
        .count()
    )


def run_wide():
    left = repeated.zipWithIndex().map(lambda x: (x[1], x[0]))
    right = left.mapValues(lambda d: d["cause"])
    return left.join(right).count()


run_narrow()  # warmup so the first timed run is not the cache fill
timed("narrow  filter + map + count", run_narrow)
timed("wide    join on unique id", run_wide)

In [ ]:
sc.stop()